In [1]:
import configparser
import os

# Initialize the config parser
config = configparser.ConfigParser()
config.read('config.ini')

# Current active profile
user_profile = 'Xuting' 

# Correcting the variable names to match your config.ini keys
# These refer to the 'input =' and 'output =' lines in your file
input_base = config[user_profile]['input']
output_base = config[user_profile]['output']

print(f"Reading data from: {input_base}")
print(f"Saving results to: {output_base}")

Reading data from: /Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData
Saving results to: /Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData


In [2]:
import pyarrow.parquet as pq
import polars as pl

# --- Paths to your files ---
# We use os.path.join to combine the base directory with the filename
quotes_file = "data_quotes_2024_06_28.parquet"
trades_file = "data_trades_2024_06_28.parquet"

quotes_path = os.path.join(input_base, quotes_file)
trades_path = os.path.join(input_base, trades_file)

# --- Efficiently count rows without loading data ---
# Now quotes_path and trades_path are correctly defined
quotes_rows = pq.ParquetFile(quotes_path).metadata.num_rows
trades_rows = pq.ParquetFile(trades_path).metadata.num_rows

print(f"Quotes rows: {quotes_rows:,}")
print(f"Trades rows: {trades_rows:,}")

Quotes rows: 1,465,325,523
Trades rows: 82,479,368


In [3]:
import pyarrow.parquet as pq
from datetime import datetime

# ---- quotes file ----
# Construct the path using the base directory from your config file
quotes_file = "data_quotes_2024_06_28.parquet"
quotes_path = os.path.join(input_base, quotes_file)

pf_quotes = pq.ParquetFile(quotes_path)
print("🧾 Quotes schema:")
print(pf_quotes.schema_arrow)     # column names and types only

# ---- trades file ----
# Construct the path using the base directory from your config file
trades_file = "data_trades_2024_06_28.parquet"
trades_path = os.path.join(input_base, trades_file)

pf_trades = pq.ParquetFile(trades_path)
print("\n💹 Trades schema:")
print(pf_trades.schema_arrow)

🧾 Quotes schema:
DATE: date32[day]
TIME_M: time64[us]
EX: string
BID: double
BIDSIZ: int64
ASK: double
ASKSIZ: int64
QU_COND: string
QU_SEQNUM: int64
NATBBO_IND: string
QU_CANCEL: string
QU_SOURCE: string
SYM_ROOT: string
SYM_SUFFIX: string

💹 Trades schema:
DATE: date32[day]
TIME_M: time64[us]
EX: string
SYM_ROOT: string
SYM_SUFFIX: string
TR_SCOND: string
SIZE: int64
PRICE: double
TR_STOP_IND: string
TR_CORR: string
TR_SEQNUM: int64
TR_ID: int64
TR_SOURCE: string
TR_RF: string


In [4]:
#Trades

target_file = os.path.join(input_base, "data_trades_2024_06_28.parquet")
target_output = os.path.join(output_base, "2024_06_28/processed_output_trades/")

# %run polar_try_partitioned.py --FILE_PATH=$target_file --OUTPUT_DIR=$target_output

In [5]:
#Quotes
quotes_input_file = os.path.join(input_base, "data_quotes_2024_06_28.parquet")
quotes_output_dir = os.path.join(output_base, "2024_06_28/processed_output_quotes/")

# Ensure the output directory exists to avoid errors
os.makedirs(quotes_output_dir, exist_ok=True)

# %run polar_try_partitioned.py --FILE_PATH=$quotes_input_file --OUTPUT_DIR=$quotes_output_dir

In [6]:
pl.Config.set_tbl_cols(50) 
pl.Config.set_tbl_width_chars(200)
pl.Config.set_tbl_rows(200) #polars.Config.tbl_rows = 50

polars.config.Config

In [7]:
# Input file remains the same
trades_input = os.path.join(input_base, "data_trades_2024_06_28.parquet")

# Output file with the "_upper" suffix
trades_output_upper = os.path.join(input_base, "data_trades_2024_06_28_upper.parquet")

%run makesTradesColsUpper.py --INPUT=$trades_input --OUTPUT=$trades_output_upper

pf_trades = pq.ParquetFile(trades_input)
print("\n💹 Trades schema:")
print(pf_trades.schema_arrow)

Uppercasing column names (preserve types): 100%|██████████| 666/666 [00:30<00:00, 21.74it/s]

Done. Wrote: /Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/data_trades_2024_06_28_upper.parquet
Input TIME_M type: time64[us]
Output TIME_M type: time64[us]

💹 Trades schema:
DATE: date32[day]
TIME_M: time64[us]
EX: string
SYM_ROOT: string
SYM_SUFFIX: string
TR_SCOND: string
SIZE: int64
PRICE: double
TR_STOP_IND: string
TR_CORR: string
TR_SEQNUM: int64
TR_ID: int64
TR_SOURCE: string
TR_RF: string


In [8]:
from pathlib import Path
import pyarrow.parquet as pq
import os

input_file_upper = os.path.join(input_base, "data_trades_2024_06_28_upper.parquet")
output_dir_clean = os.path.join(output_base, "2024_06_28/processed_output_trades_upper_clean_from_single/")

os.makedirs(output_dir_clean, exist_ok=True)


%run rewrite2.py --IN_FILE=$input_file_upper --OUT_DIR=$output_dir_clean --BATCH_ROWS=25000


DIR = Path(output_dir_clean)

files = sorted(DIR.glob("*.parquet"))
if not files:
    raise RuntimeError(f"No parquet files found in {DIR}")

# Check the schema of the first generated file to verify results
for f in files[:1]:
    arrow_schema = pq.read_schema(f)
    print(f"File Name: {f.name}")
    print(f"Schema:\n{arrow_schema}")


Rewrite2 (single->chunks): 100%|██████████| 3300/3300 [00:39<00:00, 82.75it/s]



Done. Written chunks: 3300

Row count check:
  IN_FILE : 82479368
  OUT_DIR : 82479368
  DIFF    : 0
Sanity check passed: scan_parquet works on rewritten chunks.
File Name: chunk_000001.parquet
Schema:
DATE: date32[day]
TIME_M: time64[us]
EX: string
SYM_ROOT: string
SYM_SUFFIX: string
TR_SCOND: string
SIZE: int64
PRICE: double
TR_STOP_IND: string
TR_CORR: string
TR_SEQNUM: int64
TR_ID: int64
TR_SOURCE: string
TR_RF: string


In [9]:
trades_clean_dir = os.path.join(output_base, "2024_06_28/processed_output_trades_upper_clean_from_single")
quotes_processed_dir = os.path.join(output_base, "2024_06_28/processed_output_quotes")

analysis_output_file = os.path.join(output_base, "2024_06_28/taq_analysis_output.txt")

%run query.py --TRADES_UPPER=$trades_clean_dir --QUOTES_OLD=$quotes_processed_dir --OUTPUT_FILE=$analysis_output_file


[INFO] Deleted existing output file before processing: /Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2024_06_28/taq_analysis_output.txt

Analysis successfully executed and results written to /Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2024_06_28/taq_analysis_output.txt


/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQCode/query.py:61: DeprecationWarning: `LazyFrame.fetch` is deprecated; use `LazyFrame.collect` instead, in conjunction with a call to `head`.
  print(trades_lf.fetch(n_rows=10), file=f)
/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQCode/query.py:62: DeprecationWarning: `LazyFrame.fetch` is deprecated; use `LazyFrame.collect` instead, in conjunction with a call to `head`.
  print(quotes_lf.fetch(n_rows=10), file=f)


In [10]:
import os
import polars as pl


quotes_path = os.path.join(input_base, "data_quotes_2024_06_28.parquet")

q = pl.scan_parquet(quotes_path).filter(
    (pl.col("SYM_ROOT") == "SPY") & 
    (pl.col("SYM_SUFFIX").fill_null("") == "")
).collect()

len(q)

15850080

In [11]:
QUOTES_DIR = os.path.join(output_base, "2024_06_28/processed_output_quotes/")
q = pl.scan_parquet(f"{QUOTES_DIR}/*.parquet").filter(
    (pl.col("SYM_ROOT") == "SPY") & 
    (pl.col("SYM_SUFFIX").fill_null("") == "")
).collect()
len(q)

15850080

In [12]:
QUOTES_DIR = os.path.join(output_base, "2024_06_28/processed_output_quotes_top50/")
q = pl.scan_parquet(f"{QUOTES_DIR}/*.parquet").filter(
    (pl.col("SYM_ROOT") == "SPY") & 
    (pl.col("SYM_SUFFIX").fill_null("") == "")
).collect()
len(q)

15850080

In [13]:
trades_path = os.path.join(input_base, "data_trades_2024_06_28.parquet")

# Process data
t = pl.scan_parquet(trades_path).filter(
    (pl.col("SYM_ROOT") == "SPY") & 
    (pl.col("SYM_SUFFIX").fill_null("") == "")
).collect()

len(t)

493035

In [14]:
trades_upper_path = os.path.join(input_base, "data_trades_2024_06_28_upper.parquet")

t = pl.scan_parquet(trades_upper_path).filter(
    (pl.col("SYM_ROOT") == "SPY") & 
    (pl.col("SYM_SUFFIX").fill_null("") == "")
).collect()

len(t)

493035

In [15]:
TRADES_DIR = os.path.join(output_base, "2024_06_28/processed_output_trades/")

q = pl.scan_parquet(f"{TRADES_DIR}/*.parquet").filter(
    (pl.col("SYM_ROOT") == "SPY") & 
    (pl.col("SYM_SUFFIX").fill_null("") == "")
).collect()

len(q)

493035

In [16]:
import polars as pl

schema = {
    "DATE": pl.Date,
    "TIME_M": pl.Time,
    "EX": pl.Utf8,
    "SYM_ROOT": pl.Utf8,
    "SYM_SUFFIX": pl.Utf8,
    "TR_SCOND": pl.Utf8,
    "SIZE": pl.Int64,
    "PRICE": pl.Float64,
    "TR_STOP_IND": pl.Utf8,
    "TR_CORR": pl.Utf8,
    "TR_SEQNUM": pl.Int64,
    "TR_ID": pl.Int64,
    "TR_SOURCE": pl.Utf8,
    "TR_RF": pl.Utf8,
}

TRADES = os.path.join(input_base, "data_trades_2024_06_28_upper.parquet")

t = pl.scan_parquet(TRADES, schema=schema).filter(
    (pl.col("SYM_ROOT") == "SPY") & 
    (pl.col("SYM_SUFFIX").fill_null("") == "")
).collect()
print(len(t))

t = pl.scan_parquet(TRADES, schema=schema).collect()
print(len(t))

493035
82479368


In [17]:
import polars as pl

schema = {
    "DATE": pl.Date,
    "TIME_M": pl.Time,
    "EX": pl.Utf8,
    "SYM_ROOT": pl.Utf8,
    "SYM_SUFFIX": pl.Utf8,
    "TR_SCOND": pl.Utf8,
    "SIZE": pl.Int64,
    "PRICE": pl.Float64,
    "TR_STOP_IND": pl.Utf8,
    "TR_CORR": pl.Utf8,
    "TR_SEQNUM": pl.Int64,
    "TR_ID": pl.Int64,
    "TR_SOURCE": pl.Utf8,
    "TR_RF": pl.Utf8,
}



# TRADES_CLEAN = os.path.join(output_base, "2024_06_28/processed_output_trades_upper_clean/")
# t_clean_spy = pl.scan_parquet(TRADES_CLEAN, schema=schema).filter(
#   (pl.col("SYM_ROOT") == "SPY") & (pl.col("SYM_SUFFIX").fill_null("") == "")
# ).collect()
# print(f"Clean SPY: {len(t_clean_spy)}")

TRADES_SINGLE = os.path.join(output_base, "2024_06_28/processed_output_trades_upper_clean_from_single/")


t_single_spy = pl.scan_parquet(TRADES_SINGLE).filter(
    (pl.col("SYM_ROOT") == "SPY") & (pl.col("SYM_SUFFIX").fill_null("") == "")
).collect()
print(f"Single SPY: {len(t_single_spy)}")


t_single_all = pl.scan_parquet(TRADES_SINGLE).collect()
print(f"Single Total: {len(t_single_all)}")

Single SPY: 493035
Single Total: 82479368


In [18]:
# %run offending_datatypes.py --TRADES_DIR=/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2024_06_28/processed_output_trades_upper_clean_from_single

In [19]:
%run query.py  --TRADES_UPPER="/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2024_06_28/processed_output_trades_upper_clean_from_single/*.parquet" --QUOTES_OLD="/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2024_06_28/processed_output_quotes/*.parquet" --OUTPUT_FILE="/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2024_06_28/taq_analysis_output.txt"
trades_clean = os.path.join(output_base, "2024_06_28/processed_output_trades_upper_clean_from_single/*.parquet")
quotes_old = os.path.join(output_base, "2024_06_28/processed_output_quotes/*.parquet")
output_txt = os.path.join(output_base, "2024_06_28/taq_analysis_output.txt")

%run query.py --TRADES_UPPER=$trades_clean --QUOTES_OLD=$quotes_old --OUTPUT_FILE=$output_txt

[INFO] Deleted existing output file before processing: /Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2024_06_28/taq_analysis_output.txt

Analysis successfully executed and results written to /Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2024_06_28/taq_analysis_output.txt
[INFO] Deleted existing output file before processing: /Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2024_06_28/taq_analysis_output.txt


/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQCode/query.py:61: DeprecationWarning: `LazyFrame.fetch` is deprecated; use `LazyFrame.collect` instead, in conjunction with a call to `head`.
  print(trades_lf.fetch(n_rows=10), file=f)
/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQCode/query.py:62: DeprecationWarning: `LazyFrame.fetch` is deprecated; use `LazyFrame.collect` instead, in conjunction with a call to `head`.
  print(quotes_lf.fetch(n_rows=10), file=f)
/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQCode/query.py:61: DeprecationWarning: `LazyFrame.fetch` is deprecated; use `LazyFrame.collect` instead, in conjunction with a call to `head`.
  print(trades_lf.fetch(n_rows=10), file=f)
/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQCode/query.py:62: DeprecationWarning: `LazyFrame.fetch` is deprecated; use `LazyFrame.collect` instead, in conjunction with a call to `head`.
  print(quotes_lf.fetch(n_rows=10), file=f)



Analysis successfully executed and results written to /Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2024_06_28/taq_analysis_output.txt


In [20]:
# %run stats.py --TRADES_DIR="/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2024_06_28/processed_output_trades_upper_clean_from_single/" --QUOTES_DIR="/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2024_06_28/processed_output_quotes/" --OUT_DIR="/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2024_06_28/stats_out_simple/"
# t_dir = os.path.join(output_base, "2024_06_28/processed_output_trades_upper_clean_from_single/")
# q_dir = os.path.join(output_base, "2024_06_28/processed_output_quotes/")
# o_dir = os.path.join(output_base, "2024_06_28/stats_out_simple/")

# %run stats.py --TRADES_DIR=$t_dir --QUOTES_DIR=$q_dir --OUT_DIR=$o_dir

In [21]:
OUT_DIR = os.path.join(output_base, "2024_06_28/stats_out_simple")
TRADES_OUT_CSV = os.path.join(OUT_DIR, "top50_trades_by_volume.csv")
QUOTES_OUT_CSV = os.path.join(OUT_DIR, "top50_quotes_by_rows.csv")

print(f"Reading {TRADES_OUT_CSV} and {QUOTES_OUT_CSV} …")
trades_df = pl.read_csv(TRADES_OUT_CSV)
quotes_df = pl.read_csv(QUOTES_OUT_CSV)

print("\n🔥 Top Trades (Sorted by TRADES_ROWS):")
print(trades_df.sort("TRADES_ROWS", descending=True))

print("\n📊 Top Quotes (Sorted by QUOTES_ROWS):")
print(quotes_df.sort("QUOTES_ROWS", descending=True))

Reading /Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2024_06_28/stats_out_simple/top50_trades_by_volume.csv and /Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2024_06_28/stats_out_simple/top50_quotes_by_rows.csv …

🔥 Top Trades (Sorted by TRADES_ROWS):
shape: (50, 4)
┌──────────┬────────────┬─────────────┬────────────┐
│ SYM_ROOT ┆ SYM_SUFFIX ┆ TRADES_ROWS ┆ TRADES_VOL │
│ ---      ┆ ---        ┆ ---         ┆ ---        │
│ str      ┆ str        ┆ i64         ┆ i64        │
╞══════════╪════════════╪═════════════╪════════════╡
│ NVDA     ┆            ┆ 1766952     ┆ 352104640  │
│ TSLA     ┆            ┆ 1064932     ┆ 101115275  │
│ NKE      ┆            ┆ 1027401     ┆ 151203714  │
│ AAPL     ┆            ┆ 729941      ┆ 106036016  │
│ AMZN     ┆            ┆ 605552      ┆ 92264788   │
│ SPY      ┆            ┆ 493035      ┆ 86041761   │
│ LUCY     ┆            ┆ 470271      ┆ 365583280  │
│ RIVN     ┆            ┆ 310280      ┆ 99559136   │
│ ASNS     ┆            ┆ 2

In [22]:
# %run top50_trades_corresponding_quotes.py --OUT_DIR="/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2024_06_28/stats_out_simple/" --TRADES_TOP50_CSV=top50_trades_by_volume.csv --TRADES_SRC_DIR="/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2024_06_28/processed_output_trades_upper_clean_from_single/" --TRADES_OUT_DIR="/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2024_06_28/processed_output_trades_upper_top50/" --QUOTES_SRC_DIR="/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2024_06_28/processed_output_quotes/" --QUOTES_OUT_DIR="/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2024_06_28/processed_output_quotes_top50/" --MERGED_OUT_CSV="merged_top50_sorted.csv"
# import os


# stats_dir = os.path.join(output_base, "2024_06_28/stats_out_simple/")
# top50_csv = "top50_trades_by_volume.csv"

# trades_src = os.path.join(output_base, "2024_06_28/processed_output_trades_upper_clean_from_single/")
# trades_out = os.path.join(output_base, "2024_06_28/processed_output_trades_upper_top50/")

# quotes_src = os.path.join(output_base, "2024_06_28/processed_output_quotes/")
# quotes_out = os.path.join(output_base, "2024_06_28/processed_output_quotes_top50/")

# os.makedirs(trades_out, exist_ok=True)
# os.makedirs(quotes_out, exist_ok=True)

# # --- 2. Run the Top 50 Processing Script ---
# %run top50_trades_corresponding_quotes.py \
#     --OUT_DIR=$stats_dir \
#     --TRADES_TOP50_CSV=$top50_csv \
#     --TRADES_SRC_DIR=$trades_src \
#     --TRADES_OUT_DIR=$trades_out \
#     --QUOTES_SRC_DIR=$quotes_src \
#     --QUOTES_OUT_DIR=$quotes_out \
#     --MERGED_OUT_CSV="merged_top50_sorted.csv"

In [23]:
# %run top50_trades_corresponding_quotes_persist.py --TRADES_DIR="/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2024_06_28/processed_output_trades_upper_top50/" --QUOTES_DIR="/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2024_06_28/processed_output_quotes_top50/" --STATS_DIR="/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2024_06_28/stats_out_simple/" --TOP50_CSV="top50_trades_by_volume.csv" --OUT_DIR="/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2024_06_28/merged_output_top50/" --OUT_FILE="merged_all.parquet" --CHUNK_SIZE=25000

# t_dir = os.path.join(output_base, "2024_06_28/processed_output_trades_upper_top50/")
# q_dir = os.path.join(output_base, "2024_06_28/processed_output_quotes_top50/")
# s_dir = os.path.join(output_base, "2024_06_28/stats_out_simple/")
# out_dir = os.path.join(output_base, "2024_06_28/merged_output_top50/")

# %run top50_trades_corresponding_quotes_persist.py \
#    --TRADES_DIR=$t_dir \
#    --QUOTES_DIR=$q_dir \
#    --STATS_DIR=$s_dir \
#    --TOP50_CSV="top50_trades_by_volume.csv" \
#    --OUT_DIR=$out_dir \
#    --OUT_FILE="merged_all.parquet" \
#    --CHUNK_SIZE=25000

In [26]:
%run ms.py --MERGED_FILE="/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2024_06_28/merged_output_top50/merged_all.parquet" --OUT_DIR="/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2024_06_28/ms_out/" --TICKERS=VERB,FSR,BMY


merged_file = os.path.join(output_base, "2024_06_28/merged_output_top50/merged_all.parquet")
ms_out_dir = os.path.join(output_base, "2024_06_28/ms_out/")

%run ms.py --MERGED_FILE=$merged_file --OUT_DIR=$ms_out_dir --TICKERS=AAPL,SPY,TSLA

Null TR_SEQNUM : 0
Null QU_SEQNUM : 243162
Zero BID and ASK : 51141

🔎 Running microstructure analysis for: VERB
shape: (1, 19)
┌────────┬────────┬────────────┬──────┬─────────────┬─────────────┬─────────────┬─────────────┬────────────┬────────────┬─────────────┬─────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┐
│ TICKER ┆ N_ROWS ┆ TOTAL_SIZE ┆ VWAP ┆ AVG_SPREAD_ ┆ AVG_DEPTH_T ┆ IS_MEAN_BPS ┆ IS_MEDIAN_B ┆ IS_MIN_BPS ┆ IS_MAX_BPS ┆ MI_MEAN_BPS ┆ MI_MEDIAN_B ┆ MI_MIN_BPS ┆ MI_MAX_BPS ┆ MR_MEAN_BP ┆ MR_MEDIAN_ ┆ MR_MIN_BPS ┆ MR_MAX_BPS ┆ FWD_TRADES │
│ ---    ┆ ---    ┆ ---        ┆ ---  ┆ BPS         ┆ OB          ┆ ---         ┆ PS          ┆ ---        ┆ ---        ┆ ---         ┆ PS          ┆ ---        ┆ ---        ┆ S          ┆ BPS        ┆ ---        ┆ ---        ┆ _FOR_MR    │
│ str    ┆ u32    ┆ i64        ┆ f64  ┆ ---         ┆ ---         ┆ f64         ┆ ---         ┆ f64        ┆ f64        ┆ f64         ┆ ---         ┆

In [27]:
summary_df

TICKER,N_ROWS,TOTAL_SIZE,VWAP,AVG_SPREAD_BPS,AVG_DEPTH_TOB,IS_MEAN_BPS,IS_MEDIAN_BPS,IS_MIN_BPS,IS_MAX_BPS,MI_MEAN_BPS,MI_MEDIAN_BPS,MI_MIN_BPS,MI_MAX_BPS,MR_MEAN_BPS,MR_MEDIAN_BPS,MR_MIN_BPS,MR_MAX_BPS,FWD_TRADES_FOR_MR
str,u32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i32
"""AAPL""",720117,81611225,212.61712,3199.89835,5.406072,5.68526,0.464706,0.0,1620.036587,0.0,0.0,-0.0,0.0,-4.514397,-0.233634,-1620.843646,1183.355307,10
"""SPY""",490214,79797864,545.801099,860.674318,8.293623,0.34797,0.091781,0.0,628.795375,0.0,0.0,-0.0,0.0,-0.212706,-2.0862e-12,-628.819775,591.601486,10
"""TSLA""",1052169,99395291,199.256425,8598.867672,11.875659,14.406643,1.264382,0.0,2711.116835,0.0,-0.0,-0.0,0.0,-11.878493,-0.761402,-2745.243231,1932.150079,10
